# Composite Component Initialization Regression

Regression test for the two-phase composite initialization lifecycle (`createSubComponents()` / `initializeParentFromNodesAndTerminals()`, see `docs/hugo/content/en/docs/Overview/subcomponents.md`). Each section builds the same circuit twice: once with a composite component, once with the equivalent plain elements parametrized from the same closed-form formulas the composite derives internally in `initializeParentFromNodesAndTerminals()`. A future regression that moves a frequency- or terminal-dependent value computation back into `createSubComponents()`, where it cannot see the real simulation frequency or terminal data, would desynchronize the two circuits and fail these assertions.

The compared quantities are MNA-solved observables: a node voltage for the load case and the high-voltage-side current for the transformer case. These come straight out of the system solve, so they are independent of the per-component `i_intf` bookkeeping (whose initial value needs a power-flow pre-init to be finite) and of the dependency-graph scheduler, which prunes the post-step of a purely resistive branch when nothing downstream consumes its current.

In [ ]:
import numpy as np
import dpsimpy

epsilon = 1e-6
frequency = 50
omega = 2 * np.pi * frequency
time_step = 1e-4
final_time = 0.02

## DP `PiLine` + `RXLoad`: load impedance from the Phase B closed form

A hand-built (non-CIM) two-bus DP circuit: a slack voltage source feeds a `PiLine` into an `RXLoad`. The load is parametrized explicitly with active and reactive power, so `RXLoad::initializeParentFromNodesAndTerminals()` must derive the parallel resistor and inductor values `R = V_nom^2 / P` and `L = V_nom^2 / Q / omega` using the real simulation frequency, then create and wire those sub-components. The plain-element circuit hard-codes the same closed-form values. If the composite derives or wires them differently, the voltage at the load node diverges.

In [ ]:
v_source = 10000.0
line_resistance = 0.5
line_inductance = 1e-3
load_active_power = 1e6
load_reactive_power = 2e5

load_resistance = v_source**2 / load_active_power
load_reactance = v_source**2 / load_reactive_power
load_inductance = load_reactance / omega

In [ ]:
def build_rxload_composite():
    n1 = dpsimpy.dp.SimNode("n1")
    n2 = dpsimpy.dp.SimNode("n2")
    gnd = dpsimpy.dp.SimNode.gnd

    vs = dpsimpy.dp.ph1.VoltageSource("v_s")
    vs.set_parameters(complex(v_source, 0))
    vs.connect([gnd, n1])

    line = dpsimpy.dp.ph1.PiLine("line")
    line.set_parameters(line_resistance, line_inductance)
    line.connect([n1, n2])

    load = dpsimpy.dp.ph1.RXLoad("load")
    load.set_parameters(load_active_power, load_reactive_power, v_source)
    load.connect([n2])

    system = dpsimpy.SystemTopology(frequency, [n1, n2], [vs, line, load])

    sim_name = "T496_RXLoad_Composite"
    dpsimpy.Logger.set_log_dir("logs/" + sim_name)
    sim = dpsimpy.Simulation(sim_name)
    sim.set_system(system)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.set_domain(dpsimpy.Domain.DP)
    sim.do_init_from_nodes_and_terminals(True)
    sim.run()

    return n2.attr("v").get()[0, 0]


v_load_composite = build_rxload_composite()

In [ ]:
def build_rxload_elements():
    n1 = dpsimpy.dp.SimNode("n1")
    n2 = dpsimpy.dp.SimNode("n2")
    gnd = dpsimpy.dp.SimNode.gnd

    vs = dpsimpy.dp.ph1.VoltageSource("v_s")
    vs.set_parameters(complex(v_source, 0))
    vs.connect([gnd, n1])

    line = dpsimpy.dp.ph1.PiLine("line")
    line.set_parameters(line_resistance, line_inductance)
    line.connect([n1, n2])

    load_res = dpsimpy.dp.ph1.Resistor("load_res")
    load_res.set_parameters(load_resistance)
    load_res.connect([gnd, n2])

    load_ind = dpsimpy.dp.ph1.Inductor("load_ind")
    load_ind.set_parameters(load_inductance)
    load_ind.connect([gnd, n2])

    system = dpsimpy.SystemTopology(frequency, [n1, n2], [vs, line, load_res, load_ind])

    sim_name = "T496_RXLoad_Elements"
    dpsimpy.Logger.set_log_dir("logs/" + sim_name)
    sim = dpsimpy.Simulation(sim_name)
    sim.set_system(system)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.set_domain(dpsimpy.Domain.DP)
    sim.do_init_from_nodes_and_terminals(True)
    sim.run()

    return n2.attr("v").get()[0, 0]


v_load_elements = build_rxload_elements()

In [ ]:
rxload_error = abs(v_load_composite - v_load_elements)
print(f"RXLoad composite load voltage: {v_load_composite}")
print(f"RXLoad elements load voltage:  {v_load_elements}")
print(f"Absolute error:                {rxload_error}")
assert np.isfinite(v_load_composite)
assert rxload_error < epsilon

## DP `Transformer`: snubber impedance from the Phase B closed form

The same idea applied to `DP::Ph1::Transformer`: its internal snubber resistor and capacitor values are derived in `initializeParentFromNodesAndTerminals(frequency)` from `P_SNUB_TRANSFORMER` / `Q_SNUB_TRANSFORMER` and the real simulation frequency. A regression that moved this back into `createSubComponents()` would snub at the wrong value, or at whatever stale frequency happened to be in `mFrequencies` at topology-creation time.

The plain-element circuit mirrors the composite's internal wiring (high-voltage snubber resistor, series R and L, the medium-voltage snubber referred to the high-voltage side, and the referred load). Because the composite contains an ideal ratio, its medium-voltage node sits at a different level than the referred-equivalent node, so the two circuits are compared at the high-voltage-side current drawn through the series branch. That current flows through the series inductor, a dynamic component whose post-step is always scheduled, so the comparison is unaffected by scheduler pruning.

In [ ]:
voltage_hv_side = 100000.0
voltage_mv_side = 10000.0
trafo_resistance = 1.0
trafo_inductance = 0.1
trafo_power = 1e6
load_resistance_hv_side = 10000.0

ratio = voltage_hv_side / voltage_mv_side
load_resistance_mv_side = load_resistance_hv_side / ratio**2

p_snub = dpsimpy.P_SNUB_TRANSFORMER * trafo_power
q_snub = dpsimpy.Q_SNUB_TRANSFORMER * trafo_power
snubber_resistance_hv = voltage_hv_side**2 / p_snub
snubber_resistance_mv_to_hv = ratio**2 * voltage_mv_side**2 / p_snub
snubber_capacitance_mv_to_hv = q_snub / (ratio**2 * voltage_mv_side**2) / omega

In [ ]:
def build_transformer_composite():
    n1 = dpsimpy.dp.SimNode("n1")
    n2 = dpsimpy.dp.SimNode("n2")
    gnd = dpsimpy.dp.SimNode.gnd

    vs = dpsimpy.dp.ph1.VoltageSource("v_s")
    vs.set_parameters(complex(voltage_hv_side, 0))
    vs.connect([gnd, n1])

    trafo = dpsimpy.dp.ph1.Transformer(
        "trafo", "trafo", dpsimpy.LogLevel.off, with_resistive_losses=True
    )
    trafo.set_parameters(
        voltage_hv_side,
        voltage_mv_side,
        trafo_power,
        ratio,
        0,
        trafo_resistance,
        trafo_inductance,
    )
    trafo.connect([n1, n2])

    load = dpsimpy.dp.ph1.Resistor("load")
    load.set_parameters(load_resistance_mv_side)
    load.connect([n2, gnd])

    system = dpsimpy.SystemTopology(frequency, [n1, n2], [vs, trafo, load])

    sim_name = "T496_Transformer_Composite"
    dpsimpy.Logger.set_log_dir("logs/" + sim_name)
    sim = dpsimpy.Simulation(sim_name)
    sim.set_system(system)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.set_domain(dpsimpy.Domain.DP)
    sim.do_init_from_nodes_and_terminals(True)
    sim.run()

    return trafo.attr("i_intf").get()[0, 0]


i_trafo_composite = build_transformer_composite()

In [ ]:
def build_transformer_elements():
    n1 = dpsimpy.dp.SimNode("n1")
    n2 = dpsimpy.dp.SimNode("n2")
    vn1 = dpsimpy.dp.SimNode("vn1")
    gnd = dpsimpy.dp.SimNode.gnd

    vs = dpsimpy.dp.ph1.VoltageSource("v_s")
    vs.set_parameters(complex(voltage_hv_side, 0))
    vs.connect([gnd, n1])

    snub_res_hv = dpsimpy.dp.ph1.Resistor("snub_res_hv")
    snub_res_hv.set_parameters(snubber_resistance_hv)
    snub_res_hv.connect([n1, gnd])

    series_res = dpsimpy.dp.ph1.Resistor("series_res")
    series_res.set_parameters(trafo_resistance)
    series_res.connect([n1, vn1])

    series_ind = dpsimpy.dp.ph1.Inductor("series_ind")
    series_ind.set_parameters(trafo_inductance)
    series_ind.connect([vn1, n2])

    snub_res_mv = dpsimpy.dp.ph1.Resistor("snub_res_mv")
    snub_res_mv.set_parameters(snubber_resistance_mv_to_hv)
    snub_res_mv.connect([n2, gnd])

    snub_cap_mv = dpsimpy.dp.ph1.Capacitor("snub_cap_mv")
    snub_cap_mv.set_parameters(snubber_capacitance_mv_to_hv)
    snub_cap_mv.connect([n2, gnd])

    load = dpsimpy.dp.ph1.Resistor("load")
    load.set_parameters(load_resistance_hv_side)
    load.connect([n2, gnd])

    system = dpsimpy.SystemTopology(
        frequency,
        [n1, n2, vn1],
        [vs, snub_res_hv, series_res, series_ind, snub_res_mv, snub_cap_mv, load],
    )

    sim_name = "T496_Transformer_Elements"
    dpsimpy.Logger.set_log_dir("logs/" + sim_name)
    sim = dpsimpy.Simulation(sim_name)
    sim.set_system(system)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.set_domain(dpsimpy.Domain.DP)
    sim.do_init_from_nodes_and_terminals(True)
    sim.run()

    return series_ind.attr("i_intf").get()[0, 0]


i_trafo_elements = build_transformer_elements()

In [ ]:
i_trafo_error = abs(i_trafo_composite - i_trafo_elements)
print(f"Transformer composite HV current: {i_trafo_composite}")
print(f"Transformer elements HV current:  {i_trafo_elements}")
print(f"Absolute error:                   {i_trafo_error}")
assert np.isfinite(i_trafo_composite)
assert i_trafo_error < epsilon